# 02 — Utilidad multiobjetivo, precisión y costo de los predictores

La comparación principal mide HV/AUC con respuestas históricas observadas después de seleccionar diseños en catálogos comunes. Precisión, estabilidad entre ajustes, cobertura y costo complementan esa utilidad. Este notebook solo presenta artefactos existentes.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = {"png", "svg"}
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from truss_research.evaluacion.presentacion import notebook_context

context = notebook_context()
CONFIG = context.config
STUDY_DIR, REPORT_DIR = context.study_dir, context.report_dir
DATA_DIR = CONFIG.data_dir
display(
    Markdown(
        f"**Estudio:** `{STUDY_DIR.name}` — **etiqueta:** `{CONFIG.label}` — **informe:** `{REPORT_DIR.name}`"
    )
)
if CONFIG.label == "comprobacion":
    display(
        Markdown(
            "**Comprobación técnica: sus métricas no constituyen resultados de tesis.**"
        )
    )

## 1. Configuración y familia congelada

Las familias y presupuestos se leen del estudio. Los transformadores se ajustan con entrenamiento; las OOF progresivas excluyen generaciones futuras de selección y ajuste y dejan calentamiento sin predicciones. TabPFN usa pesos preentrenados sin Optuna ni fine-tuning; Targetwise selecciona componentes por respuesta.

La familia principal se eligió por mayor AUC observado de validación en el menor presupuesto configurado, con MAE normalizado como desempate. El MAE sigue siendo el criterio del ajuste interno. La elección se lee del cierre; no se recalcula usando prueba.

In [ ]:
display(
    pd.Series(
        {
            "familias": CONFIG.models,
            "semillas": CONFIG.seeds,
            "presupuestos": CONFIG.budgets,
            "intentos_por_búsqueda": CONFIG.trials,
            "pliegues_externos": CONFIG.outer_folds,
            "pliegues_internos": CONFIG.inner_folds,
            "selección_congelada_evaluador_representante_asistente": context.frozen[
                "selected"
            ],
        }
    )
)
utility = pd.read_csv(REPORT_DIR / "predictor_utility_summary.csv")
for split, label in (("validation", "Validación"), ("test", "Prueba")):
    table = utility.loc[utility["split"].eq(split)]
    display(table)
    table.pivot(index="budget", columns="model", values="mean_auc").plot.bar(
        figsize=(12, 5),
        ylabel="AUC observado",
        title=f"Utilidad por regresor · {label}",
    )
    plt.tight_layout()
    plt.show()

## 2. Comparaciones pareadas de utilidad

Todos los regresores comparten catálogos, inicialización y presupuesto. La calidad se calcula con respuestas observadas, aunque la decisión use predicciones. Los presupuestos efectivos se reducen al agotarse un catálogo. El oráculo usa etiquetas con el mismo ranking: no es un óptimo global ni una cota de AUC.

Los intervalos promedian semillas antes de remuestrear escenarios pareados. Son exploratorios y puntuales; catálogos superpuestos y semillas no constituyen nuevos problemas estructurales independientes.

In [ ]:
display(pd.read_csv(REPORT_DIR / "predictor_utility_paired_intervals.csv"))
display(pd.read_csv(REPORT_DIR / "prediction_utility.csv"))

## 3. Errores en sus unidades y variación entre semillas

El MAE normalizado divide por desviaciones de entrenamiento y promedia las tres respuestas con pesos iguales; no es un porcentaje. MAE, RMSE, sesgo y percentil 95 describen errores físicos. Las semillas comparten fuente y partición; algunas familias deterministas pueden producir predicciones idénticas.

In [ ]:
display(pd.read_csv(REPORT_DIR / "predictive_model_summary.csv"))
display(pd.read_csv(REPORT_DIR / "predictive_seed_metrics.csv"))
display(pd.read_csv(REPORT_DIR / "predictive_category_metrics.csv"))

## 4. Intervalos de error y cobertura

`generation_block` remuestrea bloques de generaciones tras promediar pérdidas entre semillas; `design_iid_sensitivity` es un control por diseño. Pocos bloques limitan la estabilidad. No se declara significación, equivalencia ni seguridad.

Los intervalos predictivos usan calibración separada del ajuste. La deriva temporal y las categorías ausentes limitan su interpretación; cobertura marginal no garantiza cobertura conjunta ni durante selección adaptativa.

In [ ]:
display(pd.read_csv(REPORT_DIR / "predictive_paired_errors.csv"))
display(pd.read_csv(REPORT_DIR / "category_coverage.csv"))
interval_path = REPORT_DIR / "predictive_interval_coverage.csv"
if interval_path.is_file():
    display(pd.read_csv(interval_path))
else:
    display(Markdown("Este informe no contiene intervalos predictivos exportados."))

## 5. Costo y límites de interpretación

Búsqueda de hiperparámetros, OOF y ajuste final tienen costos diferentes. No se suman fases incluidas unas en otras. La inferencia individual y por lote se informa aparte. TabPFN incorpora preentrenamiento externo cuyo costo no se midió; treinta intentos no igualan cómputo entre familias.

Un error menor no garantiza mejor selección. No se mide aceleración frente al solver original ni se certifican nuevas estructuras.

In [ ]:
display(pd.read_csv(REPORT_DIR / "predictor_costs.csv"))
display(pd.read_csv(REPORT_DIR / "predictive_inference_costs.csv"))

## 6. Interpretabilidad con SHAP

SHAP es complementario a HV/AUC observado. La etapa manual `make explain` explica las familias en su semilla representativa; las semillas adicionales de la ganadora son configurables. Usa referencia de entrenamiento y los mismos casos de validación. El piloto se conserva separado y nunca se incorpora como resultado completo.

Dummy usa contribuciones nulas, Ridge explicación lineal independiente y los árboles TreeExplainer con comprobación de reconstrucción. Los demás usan permutaciones; Targetwise se explica por componente. CUDA acelera las inferencias de sus componentes TabPFN, mientras la coordinación de SHAP permanece en CPU. El costo incluye lotes, llamadas y filas predichas, intentos, RSS y VRAM cuando corresponde.

Los métodos tienen supuestos diferentes: `tree_path_dependent` usa las frecuencias internas del árbol; `interventional`, lineal e independiente usan la referencia muestreada. Las magnitudes no clasifican la calidad de los modelos ni prueban causalidad física. La agrupación de categorías vuelve a las siete entradas originales.

In [ ]:
shap_path = REPORT_DIR / "shap_global.csv"
if shap_path.is_file():
    importance = pd.read_csv(shap_path)
    display(importance)
    winner = context.frozen["selected"][0]
    selected = importance.loc[
        importance["model"].eq(winner)
        & importance["seed"].eq(CONFIG.representative_seed)
    ]
    figure, axes = plt.subplots(1, 3, figsize=(14, 4))
    for axis, target in zip(axes, ("mass_kg", "util_ratio", "z_disp_m"), strict=True):
        values = (
            selected.loc[selected["target"].eq(target)]
            .set_index("variable")["abs_shap"]
            .sort_values()
        )
        values.plot.barh(ax=axis)
        axis.set(title=target, xlabel="Media de |SHAP| en unidades de respuesta")
    plt.tight_layout()
    plt.show()
    display(pd.read_csv(REPORT_DIR / "shap_examples.csv"))
    display(pd.read_csv(REPORT_DIR / "shap_costs.csv"))
    stability = pd.read_csv(REPORT_DIR / "shap_stability.csv")
    if not stability.empty:
        display(stability)
    else:
        display(
            Markdown(
                "No se explicaron múltiples semillas: no hay estimación de estabilidad SHAP entre ajustes."
            )
        )
else:
    display(
        Markdown(
            "SHAP pendiente: ejecutar `make explain`, revisar sus costos y generar otro informe con `make report` o `make notebooks`. Esta presentación no calcula SHAP."
        )
    )

## 7. Diagnósticos finales y costo

Se presenta la familia congelada en su semilla representativa, sin volver a seleccionarla. Los residuos usan predicho menos observado. Las fases de ajustes terminados se muestran separadas; los registros de recursos incluyen los intentos fallidos y no se suman de nuevo a las fases anidadas.

In [ ]:
from truss_research.evaluacion.figuras import (
    prediction_diagnostics,
    predictor_cost_figure,
)

for figure in prediction_diagnostics(
    REPORT_DIR, (context.frozen["selected"][0], CONFIG.representative_seed)
):
    display(figure)
    plt.close(figure)
figure = predictor_cost_figure(REPORT_DIR)
display(figure)
plt.close(figure)
display(pd.read_csv(REPORT_DIR / "resource_tasks.csv"))
display(pd.read_csv(REPORT_DIR / "research_cost.csv"))